-----

### **Examen Final Práctico: Modelado y Optimización de Redes Neuronales**


**Instrucciones:**
Este examen consta de dos partes. Cada parte contiene celdas de código base y explicaciones teóricas, seguidas de dos desafíos prácticos. Deberá completar el código en los desafíos, ejecutar los modelos y responder a las preguntas de análisis en celdas de texto (Markdown) dentro de su Jupyter Notebook.

**Evaluación:**

  * **Implementación de Código (40%):** El código para cada desafío se ejecuta sin errores y cumple con los requisitos.
  * **Análisis y Respuestas (60%):** Las respuestas a las preguntas son claras, están bien justificadas y demuestran una comprensión profunda de los conceptos.

-----

### **Parte 1: Análisis de Sentimientos con Redes Recurrentes (IMDB) 💬**

En esta sección, construiremos y evaluaremos un modelo capaz de clasificar reseñas de películas del dataset IMDB como positivas o negativas. Utilizaremos una Red Neuronal Recurrente (RNN) con celdas **LSTM (Long Short-Term Memory)**, ideales para procesar datos secuenciales como el texto.

#### **Código Base: Construcción del Modelo LSTM**

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import time

**1. Carga y Preprocesamiento de Datos**

  * **Explicación:** Cargamos el dataset IMDB, limitando el vocabulario a las 10,000 palabras más frecuentes (`num_words=10000`). Cada reseña es una secuencia de números (índices de palabras). Como las redes neuronales requieren entradas de longitud uniforme, usamos `pad_sequences` para rellenar (o truncar) todas las reseñas a una longitud fija (`maxlen`).

<!-- end list -->

In [ ]:
# Parámetros iniciales
num_words = 10000
maxlen = 256 # Longitud fija para todas las reseñas

# Carga de datos
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=num_words)

# Preprocesamiento con padding
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

print(f"Forma de x_train después del padding: {x_train.shape}")
print(f"Forma de x_test después del padding: {x_test.shape}")

**2. Construcción del Modelo Secuencial**

  * **Explicación:**
      * **`Embedding`**: Esta capa convierte los índices de palabras (números enteros) en vectores densos de un tamaño fijo (`embedding_dim`). Aprende a colocar palabras con significados similares cerca unas de otras en el espacio vectorial.
      * **`LSTM`**: Procesa la secuencia de vectores de palabras, capturando dependencias a largo plazo y manteniendo un "recuerdo" del contexto, lo cual es crucial para entender el sentimiento general de la reseña.
      * **`Dense`**: Una capa de clasificación final que toma la salida de la LSTM y predice la probabilidad de que la reseña sea positiva (salida de 1).

<!-- end list -->

In [ ]:
# Construcción del modelo
model_base = keras.Sequential([
    layers.Embedding(input_dim=num_words, output_dim=128),
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid') # Sigmoide para clasificación binaria
])

# Compilación
model_base.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

model_base.summary()

**3. Entrenamiento del Modelo Base**

  * **Explicación:** Entrenamos el modelo usando el método `fit()`. Dividimos el 20% de los datos de entrenamiento para la validación (`validation_split=0.2`) para monitorear el rendimiento en datos no vistos durante el entrenamiento y detectar el sobreajuste.

<!-- end list -->

In [ ]:
history_base = model_base.fit(x_train, y_train,
                              epochs=5,
                              batch_size=128,
                              validation_split=0.2)

-----

#### **Desafíos - Parte 1**

**Desafío 1: El Impacto de `maxlen` (15 Puntos)**
Entrene tres modelos diferentes, variando el hiperparámetro `maxlen` con valores de **100, 256 y 500**. Grafique la precisión de validación y el tiempo de entrenamiento para cada uno.

**Preguntas de Análisis:**

  * ¿Cómo afecta `maxlen` al rendimiento (precisión) y la eficiencia (tiempo de entrenamiento)?
  * ¿Por qué podría una secuencia más corta o más larga ser mejor o peor para este problema específico?

**Desafío 2: Implementación de Incrustaciones GloVe (15 Puntos)**
Implemente un segundo modelo que utilice incrustaciones GloVe pre-entrenadas (por ejemplo, `glove.6B.100d.txt`). Congele la capa de incrustación (`trainable=False`) para que sus pesos no se actualicen durante el entrenamiento. Compare su precisión de validación final con la del modelo base que aprendió sus propias incrustaciones.

**Preguntas de Análisis:**

  * ¿Cuál de los dos modelos (incrustaciones aprendidas vs. GloVe) funciona mejor en el conjunto de prueba?
  * ¿Por qué cree que uno superó al otro? ¿En qué escenarios sería preferible usar incrustaciones pre-entrenadas?

-----

### **Parte 2: Reconocimiento de Dígitos con Redes Convolucionales (MNIST) 🔢**

En esta sección, construiremos una **Red Neuronal Convolucional (CNN)** para clasificar imágenes de dígitos escritos a mano del famoso dataset MNIST. Las CNN son el estándar de la industria para tareas de visión por computadora debido a su capacidad para aprender jerarquías de características visuales.

#### **Código Base: Construcción del Modelo CNN**

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

**1. Carga y Preprocesamiento de Datos**

  * **Explicación:** Cargamos el dataset MNIST. Normalizamos los píxeles de las imágenes (dividiendo por 255) para que sus valores estén en el rango de [0, 1], lo cual estabiliza el entrenamiento. Remodelamos cada imagen a `(28, 28, 1)` para que sea compatible con las capas `Conv2D`, que esperan un canal de color (en este caso, 1 para escala de grises).

<!-- end list -->

In [ ]:
# Carga de datos
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Normalización y remodelación
train_images = train_images.astype('float32') / 255
test_images = test_images.astype('float32') / 255
train_images = np.expand_dims(train_images, -1)
test_images = np.expand_dims(test_images, -1)

# Convertir etiquetas a formato one-hot
train_labels = to_categorical(train_labels, num_classes=10)
test_labels = to_categorical(test_labels, num_classes=10)

**2. Construcción del Modelo Secuencial CNN**

  * **Explicación:**
      * **`Conv2D`**: Actúa como un detector de características. Barre un filtro (kernel) sobre la imagen para detectar patrones como bordes, esquinas y texturas.
      * **`MaxPooling2D`**: Reduce la dimensionalidad de los mapas de características, haciendo que el modelo sea más eficiente y robusto a pequeñas traslaciones del dígito en la imagen.
      * **`Flatten`**: Aplana la salida 2D de las capas convolucionales en un vector 1D para poder conectarla a las capas `Dense`.
      * **`Dense`**: Capas de clasificación que toman las características extraídas y aprenden a asociarlas con uno de los 10 dígitos.

<!-- end list -->

In [ ]:
# Construcción del modelo CNN base
cnn_base = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax') # Softmax para clasificación multiclase
])

# Compilación
cnn_base.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])
cnn_base.summary()

**3. Entrenamiento del Modelo CNN Base**

  * **Explicación:** Entrenamos el modelo CNN de manera similar al anterior, usando una porción de los datos de entrenamiento para la validación.

<!-- end list -->

In [ ]:
history_cnn_base = cnn_base.fit(train_images, train_labels,
                                epochs=5,
                                batch_size=64,
                                validation_split=0.2)

-----

#### **Desafíos - Parte 2**

**Desafío 3: Combatiendo el Sobreajuste con `Dropout` (15 Puntos)**
Añada una capa `Dropout(0.5)` después de la primera capa `Flatten()`. Vuelva a entrenar el modelo y grafique las curvas de pérdida de entrenamiento y validación para el modelo original y el nuevo.

**Preguntas de Análisis:**

  * ¿Ayuda el dropout a reducir la brecha entre las curvas de pérdida de entrenamiento y validación?
  * Explique con sus palabras cómo `Dropout` logra este efecto de regularización.

**Desafío 4: Mejora de la Robustez con Aumento de Datos (15 Puntos)**
Implemente el aumento de datos utilizando la clase `ImageDataGenerator` de Keras. Configure rotaciones leves (ej. `rotation_range=10`), desplazamientos (ej. `width_shift_range=0.1`) y zooms (ej. `zoom_range=0.1`). Entrene el modelo en los datos aumentados generados en tiempo real.

**Preguntas de Análisis:**

  * ¿Mejora la precisión de validación en comparación con el modelo base?
  * ¿Por qué esta técnica de aumento de datos hace que el modelo sea más robusto ante variaciones en los datos del mundo real?